<a href="https://colab.research.google.com/github/spbromberg/ds2002-fa26/blob/main/notebooks/03-pandas-cleaning/2026-09-23%20%E2%80%94%20Cleaning%20Clinic%20%E2%80%94%20Studio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Cleaning Clinic

**Studio — 2026-09-23 · Fall 2026**  
**Class time:** 45 minutes

---

## Write the pipeline, then defend it

Monday I made the cleaning decisions and told you what they were. Today you make them, and the output is two things: a clean frame, and a **decision log** that says what you did to whose rows and why.

The log is not paperwork. On the midterm your team will disagree about whether a refund counts, and the log is what turns that into a two-minute conversation instead of an afternoon of re-deriving numbers.

Every step below follows the same three-part shape: **do it, count what you changed, log the decision.**

In [ ]:
import pandas as pd, numpy as np
from io import StringIO
raw = '''order_id,item,category,qty,price,ts
1,Cheeseburger,Food,2,$7.50,2026-09-05T12:03:00
1,Cheeseburger,Food,2,$7.50,2026-09-05T12:03:00
2,cheese burger,food,1,7.5,09/05/2026 12:40
3,Foam Finger,Merch,NULL,12,2026-09-05 13:00:00
4,UVA T-Shirt ,Apparel,2,$24.00,2026-09-05 13:05
5,Rain Poncho,RainGear,-3,6,2026-09-05T13:20:00
6,rain poncho,rain-gear,4,$6.00,
7,,Merch,1,12,2026-09-05T14:00:00'''
df = pd.read_csv(StringIO(raw))
df

### Set up the log

Run this first. Each step calls `log()` with what happened and how many rows it touched.

In [ ]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

def show_log():
    return pd.DataFrame(DECISIONS)

raw_rows = len(df)
print('starting with', raw_rows, 'rows')

### Step 0 — take inventory

**TODO:** print the shape, the dtypes, the null count per column, and the number of exact duplicate rows. Do not skip this — the rest of the studio depends on knowing what you have.

In [ ]:
# TODO

print('shape:', df.shape)

print('\ndata types:')
print(df.dtypes)

print('\nnulls per column:')
print(df.isna().sum())

print('\nexact duplicate rows:', df.duplicated().sum())

**What is wrong with this data?** List at least five specific problems:

1. There is one exact duplicate row
2. price is stored as a string (object) instead of an integer or float
3. Prices are inconsistently formated. Some include a dollar sign and some do not
4. qty has a missing value
5. item has a missing value
6. ts has one missing timestamp and is stored as text (object) instead of a date/time
7. Data has inconsistent capitalization
8. Item names are inconsistent (cheeseburger vs cheese burger)
9. One quantity is negative (does this represent a refund?)

### Step 1 — duplicates

**TODO:** drop exact duplicate rows into a new frame called `clean`, then log how many you removed. Use `.copy()` so later assignments do not warn.

In [ ]:
removed = None   # TODO: how many duplicates were there?
clean =    None  # TODO: df with duplicates dropped, copied

# TODO: log('duplicates', 'dropped exact duplicate rows', removed)

removed = df.duplicated().sum()

clean = df.drop_duplicates().copy()

log('duplicates', 'dropped exact duplicate rows', removed)

print('rows:', len(df), '->', len(clean))

### Step 2 — price into a real number

**TODO:** strip the dollar signs and any stray whitespace, then convert to float. Assert the dtype afterward so you find out now if a stray character survived.

In [ ]:
# TODO: clean['price'] = ...

clean['price'] = (
    clean['price']
    .str.replace('$', '', regex=False)
    .str.strip()
    .astype(float)
)

# assert clean['price'].dtype == float

assert clean['price'].dtype == float

# TODO: log(...) -- note that price arrived as text

log('price', 'removed dollar signs and converted price from text to float', len(clean))

print('dtype now:', clean['price'].dtype)

### Step 3 — quantity, and two decisions

**TODO:** coerce `qty` to numeric. Then decide, separately:

- what to do with the row that has no quantity
- what to do with the refund (negative quantity)

Log each decision with its row count. There is no single right answer — there is only an answer you can defend.

In [ ]:
# TODO: clean['qty'] = pd.to_numeric(...)

clean['qty'] = pd.to_numeric(clean['qty'], errors='coerce')

missing = clean['qty'].isna().sum()   # TODO: count of NaN quantities
negative = (clean['qty'] < 0).sum()  # TODO: count of negative quantities

clean = clean[clean['qty'].notna() & (clean['qty'] > 0)].copy()
clean['qty'] = clean['qty'].astype(int)

# TODO: apply your decision, then log both separately

log('missing quantity', 'dropped rows with no quantity because revenue cannot be calculated', missing)
log('negative quantity', 'dropped refund rows to calculate gross units and gross revenue', negative)

print('missing qty:', missing)
print('negative qty:', negative)
print('rows after quantity decisions:', len(clean))


### Step 4 — categories that mean one thing

**TODO:** normalize case and punctuation, then map the remaining variants with an explicit dict. Print the unique values before and after so the collapse is visible. Log how many distinct categories you started and ended with.

In [ ]:
print('before:', sorted(clean['category'].unique()))

categories_before = clean['category'].nunique()

clean['category'] = (
    clean['category']
    .str.strip()
    .str.lower()
    .str.replace('-', '', regex=False)
)

CATEGORY_MAP = {
    'apparel': 'merch'
}

clean['category'] = clean['category'].replace(CATEGORY_MAP)

categories_after = clean['category'].nunique()

print('after: ', sorted(clean['category'].unique()))

log(
    'categories',
    f'normalized categories from {categories_before} distinct values to {categories_after}',
    categories_before - categories_after
)

### Step 5 — item names

**TODO:** same treatment for `item`. One product is spelled two ways, and one row has no item at all — decide what to do with it.

In [ ]:
print('before:', sorted(clean['item'].dropna().unique()))

missing_item = clean['item'].isna().sum()

clean = clean.dropna(subset=['item']).copy()

clean['item'] = clean['item'].str.strip().str.title()

ITEM_MAP = {
    'Cheese Burger': 'Cheeseburger'
}

clean['item'] = clean['item'].replace(ITEM_MAP)

print('after: ', sorted(clean['item'].unique()))

log(
    'missing item',
    'dropped rows with no item name because the product cannot be identified',
    missing_item
)

### Step 6 — timestamps

**TODO:** parse `ts` into real datetimes, coercing failures to `NaT`. Report how many failed. Then add an `hour` column, which is only possible once the column is a real datetime.

In [ ]:
clean['ts'] = pd.to_datetime(clean['ts'], errors='coerce', format='mixed')

failed_timestamps = clean['ts'].isna().sum()

clean['hour'] = clean['ts'].dt.hour

print('unparseable timestamps (NaT):', failed_timestamps)

log(
    'timestamps',
    'parsed timestamps and converted missing or invalid values to NaT',
    failed_timestamps
)

### Step 7 — prove it

**TODO:** write at least five assertions that would catch a regression in this pipeline. Then compute `revenue` and print the totals.

In [ ]:
assert clean.duplicated().sum() == 0
assert clean['price'].dtype == float
assert clean['qty'].notna().all()
assert (clean['qty'] > 0).all()
assert clean['item'].notna().all()
assert set(clean['category'].unique()) == {'food', 'merch', 'raingear'}
assert clean['ts'].isna().sum() == 1
assert len(clean) == 4

clean['revenue'] = clean['qty'] * clean['price']

print('rows:', len(clean))
print('units:', clean['qty'].sum())
print('revenue:', clean['revenue'].sum())
print('distinct categories:', clean['category'].nunique())

clean

### Step 8 — the decision log

**TODO:** print your log. Then answer, in the markdown cell below: which single decision moved your revenue total the most, and what is the number both ways?

In [ ]:
show_log()

**The decision that mattered most:** _..._

**Revenue with it:** _..._  **Revenue without it:** _..._

---

## Checkpoint (participation)

Report your row count and revenue after cleaning, and the one decision that moved the total most.

Work in groups if you wish, then fill in the cell below **yourself**. Paste the printed output (or a screenshot of it) into this week's **Studio Checkpoint** in Canvas by **Thursday 11:59pm ET**. One submission per person, not per group.

In [ ]:
# Checkpoint
rows_after = len(clean)           # TODO
revenue_after = clean['revenue'].sum()        # TODO
biggest_decision = 'Dropped the refund row to calculate gross revenue'    # TODO: which choice moved the number most
revenue_other_way = 76.50     # TODO: the total if you had chosen differently

print('rows after cleaning:', rows_after)
print('revenue:', revenue_after)
print('decision that mattered:', biggest_decision)
print('revenue the other way:', revenue_other_way)